## This script creates a meta data file for paip slides in a folder

In [1]:
import os
import pandas as pd
import openslide
from tqdm import tqdm

def extract_svs_metadata(folder_path):
    data = []
    # Supported svs extension
    files = [f for f in os.listdir(folder_path) if f.endswith('.svs')]
    
    if not files:
        print(f"No .svs files found in {folder_path}")
        return

    print(f"Processing {len(files)} files...")

    for filename in tqdm(files):
        file_path = os.path.join(folder_path, filename)
        
        try:
            slide = openslide.OpenSlide(file_path)
            properties = slide.properties
            
            # 1. Filename & Renamed (Original without extension)
            renamed = os.path.splitext(filename)[0]
            
            # 2. Size in MB
            size_mb = round(os.path.getsize(file_path) / (1024 * 1024), 2)
            
            # 3. Extract Label (MSIH in training_data_01_MSIH.svs)
            # Logic for paip: Split by '_' and take the last part of the filename (before .svs)
            label = renamed.split('_')[-1]
            
            # 4. Extract Technical Metadata
            # Note: Property keys can vary slightly by scanner, but these are standard:
            objective = properties.get('openslide.objective-power')
            mpp_x = properties.get('openslide.mpp-x')
            mpp_y = properties.get('openslide.mpp-y')
            metadata_summary = f"Vendor: {properties.get('openslide.vendor')}, LevelCount: {slide.level_count}"

            # Check for missing critical info
            missing_fields = []
            if not objective: missing_fields.append("Objective Power")
            if not mpp_x: missing_fields.append("MPP X")
            if not mpp_y: missing_fields.append("MPP Y")
            
            if missing_fields:
                print(f"\n[!] Warning: Missing {', '.join(missing_fields)} for {filename}")

            data.append({
                "Filename": filename,
                "Size (MB)": size_mb,
                "Metadata": metadata_summary,
                "Objective Power": objective,
                "MPP X": mpp_x,
                "MPP Y": mpp_y,
                "Label": label,
                "renamed": renamed
            })
            
            slide.close()

        except Exception as e:
            print(f"Could not process {filename}: {e}")

    # Create DataFrame and Export
    df = pd.DataFrame(data)
    output_name = "slides_metadata/paip_slide_metadata_filtered.csv"
    df.to_csv(output_name, index=False)
    print(f"\nSuccess! Metadata saved to {output_name}")

# Usage
folder_to_scan = r'D:\Aamir Gulzar\dataset\paip_data\svs_files'
extract_svs_metadata(folder_to_scan)

Processing 78 files...


100%|██████████| 78/78 [00:00<00:00, 253.25it/s]


Success! Metadata saved to slides_metadata/paip_slide_metadata_filtered.csv
